<a id="sec-0"></a>
# Risk Parity + Trend — backtest notebook for the investor prospectus

**Financial Modeling (Nova SBE) · Group project · Moodle group: Group Z**

| Member | Student number |
|---|---|
| Gabriel Depraz | XXXXX |
| Marwan El-shazly | XXXXX |
| Bruno Francisco Barth | 79727 |
| Tim Jonas Trompa | 64287 |
| Paul Kuehl | 64253 |
| Felix Schoenstein | 80555 |

**Interactive app (Streamlit):** __APP_URL__
**Source code (GitHub):** __REPO_URL__

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paul2108/04_Financial-Modelin-Group-projects/blob/main/notebook/GroupZ_RiskParity_Trend.ipynb)

This notebook reproduces **every number, table and figure** in the prospectus
*“Risk Parity + Trend: a lower-drawdown alternative to the 60/40 portfolio”*.
It follows a standard **ETL** structure:

| Step | Section | What happens |
|---|---|---|
| Extract | [1](#sec-1) | daily prices are fetched from Yahoo Finance through the `yfinance` API (no data files) |
| Transform | [2](#sec-2) | cleaning, month-end sampling, returns, risk-free rate |
| Load | [3](#sec-3) | one aligned `MarketData` object that every later step reads |
| Analysis | [4](#sec-4)–[8](#sec-8) | strategy engine, benchmarks, performance, verification, robustness |
| Export | [9](#sec-9) | figures, tables and the number register used by the LaTeX prospectus |

### How to run
* **Google Colab:** open the notebook (badge above) and choose *Runtime → Run all*.
  The first cell installs `quantstats` (not pre-installed on Colab) and a current `yfinance`.
  A full run takes about 2–3 minutes.
* **Locally:** `make venv` and then `make notebook` in the repository (see `README.md`).
  Only a local run writes the export files for the prospectus.

Yahoo Finance revises adjusted prices from time to time (e.g. when dividends are re-booked).
A re-run may therefore differ from the prospectus in the last decimal. The data fingerprint in
[Section 2](#sec-2) and the retrieval time stamp in the number register document the exact data vintage.

<a id="sec-0-1"></a>
## 0.1 Setup

In [ ]:
# Install packages that Google Colab does not ship (quantstats) and make sure yfinance is current.
# Locally the project's .venv already contains everything (see requirements.txt).
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install "quantstats==0.0.86" "yfinance>=1.8"

In [ ]:
# Mandatory packages of the assignment ...
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import fsolve
import yfinance as yf
import pandas as pd
import quantstats as qs

# ... plus statsmodels (Newey-West t-statistics) and the standard library
import statsmodels.api as sm
import json
import platform
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning, module="quantstats")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

for name, mod in [("python", platform), ("numpy", np), ("pandas", pd), ("yfinance", yf),
                  ("quantstats", qs), ("statsmodels", sm)]:
    version = platform.python_version() if name == "python" else mod.__version__
    print(f"{name:<12}{version}")

<a id="sec-0-2"></a>
## 0.2 Parameters

All assumptions of the backtest sit in this one cell. The **sample period is set explicitly**:
daily data from `START` up to (but excluding) `END`. yfinance treats `end` as exclusive, so the last
month in the sample is October 2026. The strategy parameters are fixed **in advance** from the
literature: Faber's (2007) 10-month moving average and a standard one-year (252 trading days) risk estimate.
They are not tuned on this sample. [Section 8](#sec-8) shows how sensitive the results are to them.

In [ ]:
# ---------------------------------------------------------------- parameters
START = "2006-01-01"              # first day of daily data (inclusive)
END = "2026-11-01"                # last day of daily data (exclusive) -> last month = Oct 2026

RISKY_ASSETS = ["SPY", "EFA", "IEF", "VNQ", "DBC", "GLD"]   # the six asset classes held
BENCHMARK_6040 = {"SPY": 0.60, "AGG": 0.40}                  # balanced benchmark (investable)
MARKET = "^SP500TR"               # S&P 500 total return index: market proxy for CAPM alpha/beta
RF_TICKER = "^IRX"                # 13-week US T-bill yield (% p.a.): risk-free rate and cash return
OTHER_TICKERS = ["AGG", MARKET]   # benchmark building blocks that the strategy does not hold
ALL_TICKERS = RISKY_ASSETS + OTHER_TICKERS + [RF_TICKER]

COV_WINDOW = 252                  # trading days used to estimate the covariance matrix
SMA_WINDOW = 10                   # months in the trend filter's moving average (Faber, 2007)
COST_BPS = 10.0                   # transaction cost in basis points per unit of turnover
WEIGHTING = "erc"                 # "erc" (equal risk contribution), "inverse_vol" or "equal"
NW_LAGS = 4                       # Newey-West lags for the t-statistic of alpha
PERIODS = 12                      # monthly data -> 12 periods per year

ASSET_NAMES = {
    "SPY": "US equities (S&P 500)",
    "EFA": "Developed ex-US equities",
    "IEF": "US Treasuries 7-10y",
    "VNQ": "US real estate (REITs)",
    "DBC": "Broad commodities",
    "GLD": "Gold",
    "CASH": "T-bills (cash)",
    "AGG": "US aggregate bonds",
    "^SP500TR": "S&P 500 total return",
}

REPO_URL = "https://github.com/paul2108/04_Financial-Modelin-Group-projects"
APP_URL = "__APP_URL__"
NOTEBOOK_VIEW_URL = ("https://nbviewer.org/github/paul2108/04_Financial-Modelin-Group-projects/"
                     "blob/main/notebook/GroupZ_RiskParity_Trend.ipynb")

<a id="sec-0-3"></a>
## 0.3 Helpers: number register and chart style

The prospectus does not contain a single hand-typed result. Every number, table and figure it
shows is **registered** here when it is computed. Each entry records:
* the value,
* a description,
* the notebook section that produced it,
* the data source (API and tickers),
* the time the data was retrieved.

[Section 9](#sec-9) prints the full register and exports it as `results_macros.tex`. In LaTeX the
command `\res{key}` prints the value and links it to the *Sources of figures* table in the appendix.

In [ ]:
REGISTER = []   # filled while the notebook runs, exported in Section 9


def fmt_value(value, kind):
    """Return (plain text, LaTeX) representations of a result."""
    if kind == "pct1":      # 0.0734 -> 7.3%
        txt = f"{100 * value:.1f}%"
    elif kind == "pct2":
        txt = f"{100 * value:.2f}%"
    elif kind == "pp1":     # percentage points
        txt = f"{100 * value:.1f} pp"
    elif kind == "num2":
        txt = f"{value:.2f}"
    elif kind == "num1":
        txt = f"{value:.1f}"
    elif kind == "x2":      # growth multiple
        txt = f"{value:.2f}x"
    elif kind == "bps0":
        txt = f"{value:.0f} bps"
    elif kind == "int":
        txt = f"{int(value):,}"
    elif kind == "date":
        txt = pd.Timestamp(value).strftime("%b %Y")
    elif kind == "day":
        txt = pd.Timestamp(value).strftime("%d %b %Y")
    else:                   # free text
        txt = str(value)
    tex = (txt.replace("%", r"\%").replace("x", r"$\times$") if kind == "x2" else txt.replace("%", r"\%"))
    if txt.startswith("-"):
        tex = "$-$" + tex[1:]
    return txt, tex


def register(key, value, kind, description, section, sources):
    """Store a result for the prospectus. `section` is e.g. '6.1'; `sources` lists the tickers used."""
    txt, tex = fmt_value(value, kind)
    entry = dict(key=key, value=txt, tex=tex, raw=value if np.isscalar(value) else str(value),
                 description=description, section=section, sources=list(sources))
    REGISTER[:] = [e for e in REGISTER if e["key"] != key] + [entry]
    return txt


def register_item(key, label_tex, description, section, sources):
    """Register a figure or table (its 'value' is the LaTeX cross-reference, e.g. Fig.~\\ref{fig:cum})."""
    entry = dict(key=key, value=description, tex=label_tex, raw=None,
                 description=description, section=section, sources=list(sources))
    REGISTER[:] = [e for e in REGISTER if e["key"] != key] + [entry]


# --- chart style (validated categorical palette, legends below the plot, recessive grid) ----------
COLORS = {"Strategy": "#2a78d6", "60/40": "#eb6834", "S&P 500 TR": "#1baf7a",
          "ERC (no trend)": "#eda100", "1/N (no trend)": "#6250d6"}
ASSET_COLORS = {"SPY": "#2a78d6", "EFA": "#eb6834", "IEF": "#1baf7a", "VNQ": "#eda100",
                "DBC": "#e87ba4", "GLD": "#008300", "CASH": "#c3c2b7"}
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.titlesize": 10,
    "axes.labelsize": 9, "axes.edgecolor": INK_2, "axes.labelcolor": INK, "xtick.color": INK_2,
    "ytick.color": INK_2, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
    "lines.linewidth": 1.6, "font.family": "DejaVu Sans",
})
FIG_W = 6.3   # inches = text width of the prospectus


def legend_below(ax, ncol=4):
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=ncol, frameon=False)

<a id="sec-1"></a>
## 1 Extract — daily prices from the Yahoo Finance API

**Source:** Yahoo Finance, accessed through the open-source `yfinance` package (one batched request).

**Instruments:**
* the six ETFs the strategy holds,
* `AGG` for the bond leg of the 60/40 benchmark,
* the S&P 500 total-return index `^SP500TR` (CAPM market proxy),
* the 13-week T-bill yield `^IRX` (risk-free rate).

**Adjustment:** `auto_adjust=True` returns close prices adjusted for splits **and** distributions.
Returns computed from them are therefore total returns, net of each ETF's expense ratio, which is
already deducted from the fund's NAV.

**Frequency:** daily. Month-end values are derived in [Section 2](#sec-2).

In [ ]:
# ---------------------------------------------------------------- extract
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import statsmodels.api as sm
import yfinance as yf
from scipy.optimize import fsolve


def download_prices(tickers, start, end, retries=3, pause=5.0):
    """EXTRACT: daily split- and dividend-adjusted close prices from Yahoo Finance.

    One batched yfinance call (fewer requests -> fewer rate-limit problems); `end` is exclusive.
    The columns come back in the order of `tickers` (yfinance itself sorts them alphabetically).
    """
    tickers = list(tickers)
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            raw = yf.download(tickers, start=start, end=end, interval="1d",
                              auto_adjust=True, progress=False, threads=True)
            if raw is not None and not raw.empty:
                close = raw["Close"]
                if isinstance(close, pd.Series):          # single ticker -> Series
                    close = close.to_frame(tickers[0])
                close = close.reindex(columns=tickers)
                missing = [t for t in tickers if close[t].notna().sum() == 0]
                if not missing:
                    return close
                last_error = RuntimeError(f"no data returned for {missing}")
        except Exception as exc:                          # network / rate-limit errors
            last_error = exc
        if attempt < retries:
            time.sleep(pause * attempt)
    raise RuntimeError(f"Download from Yahoo Finance failed after {retries} attempts: {last_error}")

In [ ]:
DATA_RETRIEVED_UTC = pd.Timestamp.now(tz="UTC").floor("min")
prices_raw = download_prices(ALL_TICKERS, START, END)
RETRIEVED_TXT = DATA_RETRIEVED_UTC.strftime("%Y-%m-%d %H:%M UTC")

print(f"Retrieved {RETRIEVED_TXT}")
print("Raw shape (rows, columns):", prices_raw.shape)
print("Raw date range:", prices_raw.index.min().date(), "to", prices_raw.index.max().date())
prices_raw.tail(3)

In [ ]:
# Raw-data checks: first available date and missing values per ticker
raw_check = pd.DataFrame({
    "first valid date": prices_raw.apply(lambda s: s.first_valid_index().date()),
    "last valid date": prices_raw.apply(lambda s: s.last_valid_index().date()),
    "non-missing days": prices_raw.notna().sum(),
    "missing days": prices_raw.isna().sum(),
})
raw_check

The late inception of **DBC (February 2006)** produces the missing days at the start of the sample.
All other tickers have only isolated gaps. These are mostly days on which an index (`^IRX`,
`^SP500TR`) and the ETFs do not quote at the same time. Nothing is filled at this stage.

<a id="sec-2"></a>
## 2 Transform — clean, sample at month-end, compute returns

| Step | Rule |
|---|---|
| Time zone | removed (`tz_localize(None)`) so that all series align on calendar dates |
| Duplicates | duplicate dates dropped, index sorted |
| Month-end sampling | last available daily close of each calendar month (`resample("ME").last()`). We do not use yfinance's `interval="1mo"`, whose bars are labelled at the start of the month and may include a partial month. |
| Returns | **simple** returns $R_t = P_t/P_{t-1}-1$. A portfolio's return is then exactly the weighted sum of its assets' returns, $R_{p,t}=\sum_i w_{i}R_{i,t}$. Log returns add up over time, but not across assets. |
| Daily returns for risk estimates | computed only on days on which all held ETFs trade. No price is filled forward. |
| Risk-free rate | `^IRX` is an annualised yield in percent. Monthly cash return in month $t$: $r_{f,t} = \text{IRX}_{t-1}/1200$, the yield observed at the end of the previous month. It is therefore known when the cash is invested. Isolated missing quotes are filled forward for at most 5 days. |

In [ ]:
# ---------------------------------------------------------------- transform & load
@dataclass
class MarketData:
    """LOAD: the clean, aligned data that every later step reads."""
    prices_d: pd.DataFrame   # daily adjusted close prices (all tickers incl. ^IRX yield)
    rets_d: pd.DataFrame     # daily simple returns of the tradable tickers
    prices_m: pd.DataFrame   # month-end prices of the tradable tickers
    rets_m: pd.DataFrame     # monthly simple returns, labelled with the month they are earned in
    rf_m: pd.Series          # risk-free return earned in month t (known at the end of month t-1)


def clean_prices(raw):
    """TRANSFORM: tz-naive sorted daily index without duplicates or empty rows."""
    p = raw.copy()
    p.index = pd.to_datetime(p.index)
    if p.index.tz is not None:
        p.index = p.index.tz_localize(None)
    p = p[~p.index.duplicated(keep="last")].sort_index()
    return p.dropna(how="all")


def to_month_end(prices):
    """Last available daily observation of every calendar month (labelled with the month-end date)."""
    return prices.resample("ME").last()


def rf_from_irx(irx):
    """^IRX (13-week T-bill yield, % p.a.) -> monthly cash return earned in the FOLLOWING month."""
    irx_m = to_month_end(irx.ffill(limit=5))
    return (irx_m / 1200).shift(1).rename("rf")


def build_market_data(prices_raw, risky, others, rf_ticker="^IRX"):
    """TRANSFORM + LOAD: everything the strategy, benchmarks and metrics need."""
    p = clean_prices(prices_raw)
    tradables = list(risky) + list(others)
    prices_d = p[tradables + [rf_ticker]]
    rets_d = prices_d[tradables].pct_change(fill_method=None)
    prices_m = to_month_end(prices_d[tradables])
    rets_m = prices_m.pct_change(fill_method=None)
    rf_m = rf_from_irx(prices_d[rf_ticker]).reindex(prices_m.index)
    return MarketData(prices_d=prices_d, rets_d=rets_d, prices_m=prices_m, rets_m=rets_m, rf_m=rf_m)

In [ ]:
data = build_market_data(prices_raw, RISKY_ASSETS, OTHER_TICKERS, RF_TICKER)

# The "four checks" of Lecture 3 on the transformed data: shape, date range, missing values, plot
print("Daily prices   :", data.prices_d.shape, data.prices_d.index.min().date(), "to", data.prices_d.index.max().date())
print("Month-end data :", data.prices_m.shape, data.prices_m.index.min().date(), "to", data.prices_m.index.max().date())
assert data.prices_m.index[-1] == pd.Timestamp("2026-10-31"), "last month must be October 2026"
assert data.prices_d.index[-1] <= pd.Timestamp("2026-10-31"), "no partial November data"
assert data.prices_m.index.is_month_end.all()
print("\nMissing month-end values per ticker (only before inception):")
print(data.prices_m.isna().sum().to_string())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(FIG_W, 5.2), sharex=True, height_ratios=[2.2, 1])
norm = data.prices_m[RISKY_ASSETS + ["AGG"]].apply(lambda s: s / s.dropna().iloc[0])
for col in RISKY_ASSETS + ["AGG"]:
    axes[0].plot(norm.index, norm[col], color=ASSET_COLORS.get(col, INK_2),
                 lw=1.2, label=col, ls="-" if col != "AGG" else "--")
axes[0].set_yscale("log")
axes[0].set_ylabel("Price index (first month = 1, log)")
axes[0].set_title("Check: month-end total-return prices of all ETFs")
legend_below(axes[0], ncol=7)
axes[1].plot(data.prices_d.index, data.prices_d[RF_TICKER], color=INK_2, lw=1)
axes[1].set_ylabel("^IRX yield (% p.a.)")
axes[1].set_title("Check: 13-week T-bill yield (risk-free rate)", pad=4)
plt.tight_layout()
plt.show()

**Data fingerprint.** First and last observations of every series document the data vintage.
If Yahoo revises its history, this table changes and so would the results.

In [ ]:
fingerprint = pd.DataFrame({
    "first date": data.prices_d.apply(lambda s: s.first_valid_index().date()),
    "first value": data.prices_d.apply(lambda s: s.dropna().iloc[0]).round(4),
    "last date": data.prices_d.apply(lambda s: s.last_valid_index().date()),
    "last value": data.prices_d.apply(lambda s: s.dropna().iloc[-1]).round(4),
    "daily obs": data.prices_d.notna().sum(),
})
fingerprint

<a id="sec-3"></a>
## 3 Load — the sample the analysis uses

The strategy can decide only once every input exists:
* 252 complete daily returns for the covariance matrix,
* 10 month-end prices for each moving average.

DBC, the youngest ETF, sets the start. **The first decision is taken at the close of
28 February 2007, and the first return is earned in March 2007.** Every strategy variant,
benchmark and robustness check uses exactly this window, so that all of them are evaluated
on the same months.

<a id="sec-4"></a>
## 4 Strategy engine

Every month-end $t$, at the closing price, the strategy takes three steps:

1. **Risk estimate.** Estimate the covariance matrix $\Sigma_t$ of the six ETFs from the last
   252 daily returns up to and including $t$.
2. **Risk parity.** Choose weights so that every asset class contributes the same share of
   portfolio risk (equal risk contribution, ERC; Maillard, Roncalli & Teïletche, 2010):
   $$RC_i = \frac{w_i(\Sigma w)_i}{w^\top\Sigma w} = \frac{1}{N}\qquad i=1,\dots,N .$$
   The weights have no closed form, so we solve the nonlinear system with **`scipy.optimize.fsolve`**.
   Following Spinu (2013), ERC holds exactly when $(\Sigma x)_i - \tfrac{1}{N x_i}=0$ for all $i$. These
   are the first-order conditions of a strictly convex problem, so the positive solution is unique.
   We solve them in log-coordinates $x=e^{y}$, which keeps every weight positive, and set
   $w=x/\sum_i x_i$. A test on 2,000 random covariance matrices converged every time.
   The starting value is the inverse-volatility portfolio. Every solution is checked: if the risk
   contributions are not equal to within $10^{-8}$, the month falls back to inverse-volatility
   weights and the fallback is counted.
3. **Trend filter.** An asset class is held only if its month-end price is above its 10-month
   simple moving average (Faber, 2007). Otherwise its risk-parity weight is moved into T-bills (cash).

The new weights earn the returns of month $t+1$: **decide at $t$, earn $t+1$** (the `shift(1)` rule
of Lecture 4). Between two rebalancing dates the holdings drift with prices. At each rebalancing,
the traded amount (turnover) is the distance between the target weights and the drifted weights:
$$TO_t=\sum_{i\in\text{ETFs}}\left|w_{i,t}-\tilde w_{i,t}\right|,\qquad
\tilde w_{i,t}=\frac{w_{i,t-1}(1+R_{i,t})}{1+R_{p,t}} .$$
Trading costs $c$ per unit of turnover reduce the portfolio value before the next month's return:
$V_{t+1}=V_t\,(1-c\cdot TO_t)\,(1+R_{p,t+1})$. Switching into and out of T-bills is assumed to be free,
as it is in a money-market sweep account.

In [ ]:
# ---------------------------------------------------------------- strategy: risk parity weights
def inverse_vol_weights(cov):
    """Weights proportional to 1/volatility (the ERC solution when correlations are zero)."""
    iv = 1.0 / np.sqrt(np.diag(np.asarray(cov, dtype=float)))
    return iv / iv.sum()


def risk_contributions(w, cov):
    """Share of portfolio variance contributed by each asset (sums to 1)."""
    w = np.asarray(w, dtype=float)
    cov = np.asarray(cov, dtype=float)
    return w * (cov @ w) / (w @ cov @ w)


def erc_weights(cov, tol=1e-8):
    """Equal-risk-contribution weights, solved with scipy's fsolve (Spinu, 2013).

    ERC holds when x_i (S x)_i = 1/N for all i, i.e. when (S x)_i - 1/(N x_i) = 0: the first-order
    condition of the convex problem min 0.5 x'Sx - (1/N) sum(log x_i). We solve these N equations in
    log-coordinates x = exp(y) > 0, so the weights stay positive, and return w = x / sum(x).
    Returns (weights, ok). If the solution fails the check, inverse-vol weights are returned and ok=False.
    """
    cov = np.atleast_2d(np.asarray(cov, dtype=float))
    n = cov.shape[0]
    S = cov / np.mean(np.diag(cov))                 # rescale: the solution does not depend on the scale
    iv = 1.0 / np.sqrt(np.diag(S))
    x0 = iv / np.sqrt(iv @ S @ iv)                  # inverse-vol start, scaled so that x0' S x0 = 1

    def equations(y):
        x = np.exp(y)
        return S @ x - 1.0 / (n * x)

    def jacobian(y):                                # d equations_i / d y_j
        x = np.exp(y)
        return S * x[None, :] + np.diag(1.0 / (n * x))

    y, _info, ier, _msg = fsolve(equations, np.log(x0), fprime=jacobian, full_output=True, xtol=1e-12)
    w = np.exp(y)
    w = w / w.sum()
    ok = bool(ier == 1 and np.all(np.isfinite(w))
              and np.max(np.abs(risk_contributions(w, cov) - 1.0 / n)) < tol)
    if not ok:
        return inverse_vol_weights(cov), False
    return w, True

In [ ]:
# ---------------------------------------------------------------- strategy: trend filter and target weights
def trend_signal(prices_m, window=10):
    """1.0 if the month-end price is above its `window`-month SMA (incl. the current month), 0.0 if not,
    NaN while the moving average is not yet available."""
    sma = prices_m.rolling(window, min_periods=window).mean()
    return (prices_m > sma).astype(float).where(sma.notna())


def first_decision_date(data, assets, cov_window=252, sma_window=10):
    """First month-end at which the covariance window and every moving average are complete."""
    assets = list(assets)
    px = data.prices_d[assets].dropna()
    first_cov = px.index[cov_window]                 # cov_window returns need cov_window + 1 prices
    first_sig = trend_signal(data.prices_m[assets], sma_window).dropna().index[0]
    months = data.prices_m.index
    return months[(months >= first_cov) & (months >= first_sig)][0]


def target_weights(data, assets, decision_dates, cov_window=252, sma_window=10,
                   use_trend=True, weighting="erc"):
    """Target weights decided at each month-end (rows) for the assets plus CASH (columns).

    Only information up to and including the decision date is used.
    """
    assets = list(assets)
    px = data.prices_d[assets].dropna()              # days on which all held ETFs trade
    rd = px.pct_change(fill_method=None).iloc[1:]
    sig = trend_signal(data.prices_m[assets], sma_window)
    rows, fallbacks = [], 0
    for t in decision_dates:
        window = rd.loc[:t].iloc[-cov_window:]
        if len(window) < cov_window:
            raise ValueError(f"{t.date()}: only {len(window)} daily returns available, need {cov_window}")
        cov = np.atleast_2d(np.cov(window.to_numpy(), rowvar=False))
        if weighting == "erc":
            w, ok = erc_weights(cov)
            fallbacks += int(not ok)
        elif weighting == "inverse_vol":
            w = inverse_vol_weights(cov)
        elif weighting == "equal":
            w = np.full(len(assets), 1.0 / len(assets))
        else:
            raise ValueError(f"unknown weighting '{weighting}'")
        w = pd.Series(w, index=assets, dtype=float)
        if use_trend:
            on = sig.loc[t, assets]
            if on.isna().any():
                raise ValueError(f"{t.date()}: trend signal not yet available")
            w = w.where(on == 1.0, 0.0)
        w["CASH"] = 1.0 - w.sum()
        rows.append(w.rename(t))
    W = pd.DataFrame(rows)
    W.index.name = "decision_date"
    W.attrs["fsolve_fallbacks"] = fallbacks
    return W


def constant_weights(weights, decision_dates):
    """Fixed-mix target weights (e.g. 60/40), rebalanced back to target at every decision date."""
    w = pd.Series(weights, dtype=float)
    if "CASH" not in w.index:
        w["CASH"] = 1.0 - w.sum()
    cols = [c for c in w.index if c != "CASH"] + ["CASH"]
    W = pd.DataFrame([w[cols].to_numpy()] * len(decision_dates),
                     index=pd.DatetimeIndex(decision_dates, name="decision_date"), columns=cols)
    W.attrs["fsolve_fallbacks"] = 0
    return W

In [ ]:
# ---------------------------------------------------------------- backtest engine
def backtest(weights, data, cost_bps=10.0):
    """Monthly backtest: weights decided at month-end t earn the simple returns of month t+1.

    Holdings drift between rebalancing dates. Turnover = sum over ETFs of |target - drifted|.
    Costs = cost_bps/10,000 x turnover, deducted from portfolio value before the month's return.
    CASH earns the risk-free rate. Returns a DataFrame indexed by the month the return is earned in.
    """
    cols = list(weights.columns)
    assets = [c for c in cols if c != "CASH"]
    R = data.rets_m.reindex(columns=assets).copy()
    R["CASH"] = data.rf_m
    R = R[cols]
    months = data.rets_m.index
    drifted = pd.Series(0.0, index=cols)
    drifted["CASH"] = 1.0                            # the investor starts fully in cash
    rows = []
    for t, w in weights.iterrows():
        pos = months.get_loc(t)
        if pos + 1 >= len(months):                   # last decision: no next-month return yet
            break
        t_next = months[pos + 1]
        r = R.loc[t_next]
        if r[w > 0].isna().any():
            raise ValueError(f"missing return in {t_next.date()} for a held asset")
        r = r.fillna(0.0)
        turnover = float((w[assets] - drifted[assets]).abs().sum()) if assets else 0.0
        cost = cost_bps / 1e4 * turnover
        gross = float((w * r).sum())
        net = (1.0 - cost) * (1.0 + gross) - 1.0
        rows.append({"date": t_next, "gross": gross, "net": net, "turnover": turnover,
                     "cost": cost, "cash_weight": float(w["CASH"]), "rf": float(r["CASH"])})
        drifted = w * (1.0 + r) / (1.0 + gross)
    return pd.DataFrame(rows).set_index("date")

<a id="sec-4-4"></a>
### 4.4 Performance measures

All measures use **monthly simple returns** with `ddof=1`. $R$ is the strategy's net return, $R_f$ the
T-bill return, $R_b$ the 60/40 benchmark and $R_M$ the S&P 500 total return. The definitions are
those of Lecture 3 and the course formula sheet:

| Measure | Definition | Annualisation |
|---|---|---|
| Average return | $\bar R$ | $\times 12$ (arithmetic) |
| CAGR | $\left[\prod_t(1+R_t)\right]^{12/T}-1$ | geometric; it is lower than the arithmetic mean by roughly $\sigma^2/2$ |
| Volatility | $\sigma(R)$ | $\times\sqrt{12}$ |
| Sharpe ratio | $E(R-R_f)/\sigma(R-R_f)$ | $12/\sqrt{12}=\sqrt{12}$ |
| Beta | $\beta=\operatorname{cov}(R-R_f,R_M-R_f)/\operatorname{var}(R_M-R_f)$ | **not annualised** |
| Alpha (Jensen) | $\alpha=E[R-R_f]-\beta\,E[R_M-R_f]$, with a Newey-West (4 lags) $t$-statistic | $\times 12$ |
| Tracking error | $\sigma(R-R_b)$ | $\times\sqrt{12}$ |
| Information ratio | $E(R-R_b)/\sigma(R-R_b)$, the benchmark-mandate version of Lecture 3 | $12/\sqrt{12}$ |
| Maximum drawdown | $\min_t(V_t/\max_{s\le t}V_s-1)$, including the start value $V_0=1$ | – |
| Turnover | $\overline{TO}$, share of portfolio value traded | $\times 12$ (per year) |

Beta and alpha are computed against the market, the S&P 500 (CAPM), and against the 60/40 benchmark.

In [ ]:
# ---------------------------------------------------------------- performance measures
def nav_from_returns(r, start=None):
    """Value of 1 invested at the decision date before the first return (the series starts at 1)."""
    r = r.dropna()
    if start is None:
        start = r.index[0] - pd.offsets.MonthEnd(1)
    nav = (1.0 + r).cumprod()
    return pd.concat([pd.Series([1.0], index=pd.DatetimeIndex([start])), nav]).rename(r.name)


def drawdown_series(r):
    nav = nav_from_returns(r)
    return nav / nav.cummax() - 1.0


def max_drawdown(r):
    return float(drawdown_series(r).min())


def capm_regression(excess, factor_excess, maxlags=4):
    """OLS of excess returns on factor excess returns -> (beta, monthly alpha, Newey-West t(alpha))."""
    df = pd.concat([excess, factor_excess], axis=1).dropna()
    X = sm.add_constant(df.iloc[:, 1].to_numpy())
    fit = sm.OLS(df.iloc[:, 0].to_numpy(), X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    alpha, beta = fit.params
    return float(beta), float(alpha), float(fit.tvalues[0])


def performance_summary(r, rf, benchmark=None, market=None, turnover=None, periods=12, nw_lags=4):
    """All performance measures of the prospectus for one monthly return series."""
    r = r.dropna()
    rf = rf.reindex(r.index)
    ex = r - rf
    out = {
        "months": int(len(r)),
        "ann_mean": periods * r.mean(),
        "cagr": float(np.prod(1.0 + r) ** (periods / len(r)) - 1.0),
        "ann_vol": np.sqrt(periods) * r.std(ddof=1),
        "sharpe": periods * ex.mean() / (np.sqrt(periods) * ex.std(ddof=1)),
        "max_dd": max_drawdown(r),
        "worst_12m": float(((1.0 + r).rolling(12).apply(np.prod, raw=True) - 1.0).min()),
        "final_value": float(np.prod(1.0 + r)),
    }
    if market is not None:
        b, a, t = capm_regression(ex, market.reindex(r.index) - rf, nw_lags)
        out.update(beta_mkt=b, alpha_mkt=periods * a, alpha_mkt_t=t)
    if benchmark is not None:
        bench = benchmark.reindex(r.index)
        active = r - bench
        out["tracking_error"] = np.sqrt(periods) * active.std(ddof=1)
        out["info_ratio"] = (periods * active.mean() / out["tracking_error"]
                             if out["tracking_error"] > 0 else np.nan)
        b, a, t = capm_regression(ex, bench - rf, nw_lags)
        out.update(beta_bench=b, alpha_bench=periods * a, alpha_bench_t=t)
    if turnover is not None:
        out["turnover"] = periods * turnover.reindex(r.index).mean()
    return out


def breakeven_cost(diff, x0=10.0, max_bps=1000.0):
    """Cost level c (bps per unit turnover) with diff(c) = 0, solved with fsolve.

    `diff(c)` is e.g. Sharpe(strategy, c) - Sharpe(benchmark, c). Returns (c, ok); (nan, False) if the
    strategy is not ahead even at zero cost, so that no break-even exists.
    """
    if not diff(0.0) > 0:
        return np.nan, False
    sol, _info, ier, _msg = fsolve(lambda c: [diff(float(c[0]))], [x0], full_output=True)
    c = float(sol[0])
    ok = bool(ier == 1 and 0.0 <= c <= max_bps and abs(diff(c)) < 1e-8)
    return (c if ok else np.nan), ok

In [ ]:
# ---------------------------------------------------------------- one call for notebook and app
def run_strategy(data, assets=None, cov_window=252, sma_window=10, use_trend=True, weighting="erc",
                 cost_bps=10.0, start_decision=None, end_decision=None, benchmark_weights=None):
    """Run the full strategy (and optionally a fixed-mix benchmark) over a common decision window."""
    assets = list(RISKY_ASSETS if assets is None else assets)
    if start_decision is None:
        start_decision = first_decision_date(data, assets, cov_window, sma_window)
    months = data.prices_m.index
    dates = months[months >= pd.Timestamp(start_decision)]
    if end_decision is not None:
        dates = dates[dates <= pd.Timestamp(end_decision)]
    W = target_weights(data, assets, dates, cov_window, sma_window, use_trend, weighting)
    res = backtest(W, data, cost_bps)
    out = {"weights": W, "result": res, "returns": res["net"].rename("Strategy"),
           "gross": res["gross"].rename("Strategy (gross)"), "start_decision": pd.Timestamp(dates[0]),
           "fsolve_fallbacks": W.attrs["fsolve_fallbacks"]}
    if benchmark_weights is not None:
        Wb = constant_weights(benchmark_weights, dates)
        resb = backtest(Wb, data, cost_bps)
        out["benchmark"] = resb["net"].rename("Benchmark")
        out["benchmark_result"] = resb
    return out